In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

Pesticides = pd.read_csv("/content/pesticides.csv")
Yield = pd.read_csv("/content/yield.csv")
Rainfall = pd.read_csv("/content/rainfall.csv")
Temperature = pd.read_csv("/content/temp.csv")

Yield.columns = Yield.columns.str.strip()
Pesticides.columns = Pesticides.columns.str.strip()
Rainfall.columns = Rainfall.columns.str.strip()
Temperature.columns = Temperature.columns.str.strip()

Temperature = Temperature.rename(columns={
    "country": "Area",
    "year": "Year"
})

Rainfall = Rainfall.rename(columns={
    "average_rain_fall_mm_per_year": "Average_Rainfall"
})

Yield = Yield.rename(columns={"Value": "Value_yield"})

rain_temp = Rainfall.merge(
    Temperature[["Area", "Year", "avg_temp"]],
    on=["Area", "Year"],
    how="inner"
)

print(rain_temp.head())
print(rain_temp.columns.tolist())
print(rain_temp.shape)

          Area  Year Average_Rainfall  avg_temp
0  Afghanistan  1985              327     15.52
1  Afghanistan  1986              327     14.71
2  Afghanistan  1987              327     15.46
3  Afghanistan  1989              327     14.46
4  Afghanistan  1990              327     15.45
['Area', 'Year', 'Average_Rainfall', 'avg_temp']
(8802, 4)


In [ ]:
train_rain = rain_temp[rain_temp["Year"] <= 2008].copy()
test_rain = rain_temp[rain_temp["Year"] > 2008].copy()

# Fix: Convert 'Average_Rainfall' to numeric and handle non-numeric values
train_rain['Average_Rainfall'] = pd.to_numeric(train_rain['Average_Rainfall'], errors='coerce')
test_rain['Average_Rainfall'] = pd.to_numeric(test_rain['Average_Rainfall'], errors='coerce')

# Drop rows where 'Average_Rainfall' is NaN after coercion
train_rain = train_rain.dropna(subset=['Average_Rainfall'])
test_rain = test_rain.dropna(subset=['Average_Rainfall'])

# Baseline: rainfall from Area + Year
X_train_base = pd.get_dummies(train_rain[["Area", "Year"]])
X_test_base = pd.get_dummies(test_rain[["Area", "Year"]])
X_test_base = X_test_base.reindex(columns=X_train_base.columns, fill_value=0)

y_train_rain = train_rain["Average_Rainfall"]
y_test_rain = test_rain["Average_Rainfall"]

model_base = xgb.XGBRegressor(n_estimators=100, max_depth=4, random_state=42)
model_base.fit(X_train_base, y_train_rain)
preds_base = model_base.predict(X_test_base)

mae_base = mean_absolute_error(y_test_rain, preds_base)

# With temperature: rainfall from Area + Year + avg_temp
X_train_temp = pd.get_dummies(train_rain[["Area", "Year", "avg_temp"]])
X_test_temp = pd.get_dummies(test_rain[["Area", "Year", "avg_temp"]])
X_test_temp = X_test_temp.reindex(columns=X_train_temp.columns, fill_value=0)

model_temp = xgb.XGBRegressor(n_estimators=100, max_depth=4, random_state=42)
model_temp.fit(X_train_temp, y_train_rain)
preds_temp = model_temp.predict(X_test_temp)

mae_temp = mean_absolute_error(y_test_rain, preds_temp)

print("Rainfall prediction")
print("Baseline MAE:", round(mae_base, 2))
print("With temperature MAE:", round(mae_temp, 2))

if mae_temp < mae_base:
    print("Temperature helped predict rainfall.")
else:
    print("Temperature did not help predict rainfall.")

Rainfall prediction
Baseline MAE: 83.12
With temperature MAE: 73.12
Temperature helped predict rainfall.


In [ ]:
data = Yield.merge(
    Pesticides[["Area", "Year", "Value"]],
    on=["Area", "Year"],
    how="left"
).rename(columns={"Value": "Value_pest"})

data = data.merge(
    Rainfall[["Area", "Year", "Average_Rainfall"]],
    on=["Area", "Year"],
    how="left"
)

data = data.merge(
    Temperature[["Area", "Year", "avg_temp"]],
    on=["Area", "Year"],
    how="left"
)

data = data.dropna(subset=["Value_yield", "Area", "Year"])

In [ ]:
rain_features_all = pd.get_dummies(rain_temp[["Area", "Year", "avg_temp"]])
rain_features_all = rain_features_all.reindex(columns=X_train_temp.columns, fill_value=0)

rain_temp["Predicted_Rainfall"] = model_temp.predict(rain_features_all)

data = data.merge(
    rain_temp[["Area", "Year", "Predicted_Rainfall"]].drop_duplicates(),
    on=["Area", "Year"],
    how="left"
)

print(data[["Area", "Year", "Average_Rainfall", "Predicted_Rainfall"]].head())

          Area  Year Average_Rainfall  Predicted_Rainfall
0  Afghanistan  1961              NaN                 NaN
1  Afghanistan  1962              NaN                 NaN
2  Afghanistan  1963              NaN                 NaN
3  Afghanistan  1964              NaN                 NaN
4  Afghanistan  1965              NaN                 NaN


In [ ]:
train_yield = data[data["Year"] <= 2008].copy()
test_yield = data[data["Year"] > 2008].copy()

# Model using actual rainfall
features_actual = ["Value_pest", "Year", "Item", "Area", "Average_Rainfall", "avg_temp"]
X_train_actual = pd.get_dummies(train_yield[features_actual])
X_test_actual = pd.get_dummies(test_yield[features_actual])
X_test_actual = X_test_actual.reindex(columns=X_train_actual.columns, fill_value=0)

y_train_yield = train_yield["Value_yield"]
y_test_yield = test_yield["Value_yield"]

model_actual = xgb.XGBRegressor(n_estimators=100, max_depth=4, random_state=42)
model_actual.fit(X_train_actual, y_train_yield)
preds_actual = model_actual.predict(X_test_actual)

mae_actual = mean_absolute_error(y_test_yield, preds_actual)

# Model using predicted rainfall instead
features_pred = ["Value_pest", "Year", "Item", "Area", "Predicted_Rainfall", "avg_temp"]
X_train_pred = pd.get_dummies(train_yield[features_pred])
X_test_pred = pd.get_dummies(test_yield[features_pred])
X_test_pred = X_test_pred.reindex(columns=X_train_pred.columns, fill_value=0)

model_pred = xgb.XGBRegressor(n_estimators=100, max_depth=4, random_state=42)
model_pred.fit(X_train_pred, y_train_yield)
preds_pred = model_pred.predict(X_test_pred)

mae_pred = mean_absolute_error(y_test_yield, preds_pred)

print("Yield prediction")
print("Using actual rainfall MAE:", round(mae_actual, 2))
print("Using predicted rainfall MAE:", round(mae_pred, 2))

if mae_pred < mae_actual:
    print("Predicted rainfall worked well as an input for yield.")
else:
    print("Predicted rainfall was worse than actual rainfall.")

Yield prediction
Using actual rainfall MAE: 17636.5
Using predicted rainfall MAE: 17947.16
Predicted rainfall was worse than actual rainfall.
